In [2]:
import pandas as pd
from konlpy.tag import Okt
from collections import Counter

# 1. 데이터 및 분석기 로드
df = pd.read_csv('data2023_2.csv')
okt = Okt()

# 2. 챔피언 리스트 (정규화용)
champions_list = [
    '가렌', '갈리오', '갱플랭크', '그라가스', '그레이브즈', '그웬', '나르', '나미', '나서스', '노틸러스', 
    '녹턴', '누누와 윌럼프', '니달리', '니코', '닐라', '다리우스', '다이애나', '드레이븐', '라이즈', '라칸', 
    '람머스', '럭스', '럼블', '레나타 글라스크', '레넥톤', '레오나', '렉사이', '렐', '렝가', '루시안', 
    '룰루', '르블랑', '리 신', '리븐', '리산드라', '릴리아', '마스터 이', '마오카이', '말자하', '말파이트', 
    '모데카이저', '모르가나', '문도 박사', '미스 포츈', '밀리오', '바드', '바루스', '바이', '베이가', '베인', 
    '벡스', '벨베스', '벨코즈', '볼리베어', '브라움', '브랜드', '블라디미르', '블리츠크랭크', '비에고', '빅토르', 
    '뽀삐', '사미라', '사이온', '사일러스', '샤코', '세나', '세라핀', '세주아니', '세트', '소나', 
    '소라카', '쉔', '쉬바나', '스웨인', '스카너', '스몰더', '시비르', '신 짜오', '신드라', '신지드', 
    '쓰레쉬', '아리', '아무무', '아우렐리온 솔', '아이번', '아지르', '아칼리', '아크샨', '아트록스', '아펠리오스', 
    '알리스타', '앰베사', '애니', '애니비아', '애쉬', '야스오', '에코', '엘리스', '오공', '오로라', 
    '오른', '오리아나', '올라프', '요네', '요릭', '우디르', '우르곳', '워윅', '유미', '이렐리아', 
    '이블린', '이즈리얼', '일라오이', '자르반 4세', '자야', '자이라', '자크', '잔나', '잭스', '제드', 
    '제리', '제이스', '조이', '직스', '진', '질리언', '징크스', '초가스', '카르마', '카밀', 
    '카사딘', '카서스', '카시오페아', '카이사', '카직스', '카타리나', '칼리스타', '케넨', '케이틀린', '케인', 
    '케일', '코그모', '코르키', '콰오아', '클레드', '키아나', '킨드레드', '타릭', '탈론', '탈리야', 
    '탐 켄치', '트런들', '트리스타나', '트린다미어', '트위스티드 페이트', '트위치', '티모', '파이크', '판테온', '피들스틱', 
    '피오라', '피즈', '하이머딩거', '헤카림', '흐웨이'
]
def extract_champion_nouns(text):
    # 형태소 분석으로 명사만 추출
    nouns = okt.nouns(str(text))
    # 추출된 명사 중 챔피언 리스트에 있는 것만 필터링
    return [n for n in nouns if n in champions_list]

# 3. 분석 실행
all_champs = []
for comment in df['comment_text']:
    all_champs.extend(extract_champion_nouns(comment))

# 4. 결과 집계 및 저장
result = pd.DataFrame(Counter(all_champs).most_common(), columns=['챔피언', '언급횟수'])
print(result.head(30))

     챔피언  언급횟수
0     제드   124
1     유미   104
2     샤코    99
3     가렌    94
4     요릭    92
5   다이애나    88
6     애쉬    85
7    징크스    84
8      진    83
9   아트록스    77
10   스웨인    76
11    티모    67
12    케인    66
13  이즈리얼    65
14    아리    65
15   스카너    63
16  케이틀린    63
17  다리우스    60
18   야스오    59
19   제이스    57
20    바이    57
21  이렐리아    56
22   빅토르    56
23    베인    54
24    잭스    53
25    럭스    52
26  말파이트    52
27    렝가    50
28    에코    49
29     쉔    44


In [3]:
# 포지션별 기준/ 한국어 아이템약어 기준(로아,트포,무대,데파)-한국 커뮤 제한/ 매칭 라인 선택/게임 패치 후 유저 반응(시기별 반응 레딧,유튜브)
#
#
#

In [4]:
import pandas as pd
import re
from collections import Counter

# 1. 172명 챔피언 리스트 및 감성 사전 정의 (생략 - 본문 동일)
# ... [이전 코드와 동일한 리스트 및 사전 설정] ...

def get_advanced_sentiment(comment_text):
    """댓글 하나의 감성 점수를 계산 (문맥 고려)"""
    words = re.sub(r'[^\w\s]', ' ', str(comment_text)).split()
    score = 0
    for word in words:
        if any(pk in word for pk in pos_keywords): score += 1
        if any(nk in word for nk in neg_keywords): score -= 1
    for i, word in enumerate(words):
        if any(cw in word for cw in neutral_context_words):
            context_window = " ".join(words[max(0, i-2) : min(len(words), i+3)])
            if any(pm in context_window for pm in pos_modifiers): score += 1
            if any(nm in context_window for nm in neg_modifiers): score -= 1
    return score

def is_target_mentioned(text, champ):
    """'진', '쉔' 등 1글자 챔피언의 오탐지를 방지하는 명사 판별 로직"""
    if len(champ) > 1:
        return champ in str(text).replace(' ', '')
    else:
        # 단어 앞뒤가 공백이거나 조사(이/가/은/는/을/를 등)가 붙은 명사 형태만 매칭
        pattern = f"(?:^|\\s){champ}(?=[이가은는을를의와과도만\\s]|$)"
        return bool(re.search(pattern, str(text)))

def run_champion_report(file_name):
    df = pd.read_csv(file_name)
    final_results = []
    for champ in champions_list:
        mask = df['comment_text'].apply(lambda x: is_target_mentioned(x, champ))
        champ_df = df[mask]
        if len(champ_df) < 5: continue
            
        total_score = sum(get_advanced_sentiment(t) for t in champ_df['comment_text'])
        all_words = []
        for text in champ_df['comment_text']:
            clean_words = re.sub(r'[^\w\s]', ' ', str(text)).split()
            all_words.extend([w for w in clean_words if champ not in w and len(w) > 1])
            
        keywords_str = ", ".join([f"{w}({c})" for w, c in Counter(all_words).most_common(3)])
        sentiment = "긍정적" if total_score > 0 else ("부정적" if total_score < 0 else "중립적")
        
        # 분석 결과 문장 생성
        thought = "유저들이 해당 챔피언의 디자인이나 성능 만족도, 혹은 재미 요소에 대해 좋은 평가를 내리고 있는 것으로 보입니다." if sentiment == "긍정적" else \
                  "유저들이 해당 챔피언의 밸런스 붕괴나 상대할 때의 불쾌감, 혹은 성능 하락에 대해 불만을 느끼고 있는 것으로 보입니다." if sentiment == "부정적" else \
                  "객관적인 패치 수치 공유나 일반적인 질문 사항으로 언급되고 있습니다."
            
        final_results.append({
            '챔피언': champ, '언급횟수': len(champ_df), '감성': sentiment,
            '분석결과': f"[{keywords_str}] 등의 이유로 챔피언을 언급했다. 그래서 내 생각은 {thought}"
        })
    return pd.DataFrame(final_results)

# --- 실행부 ---
if __name__ == "__main__":
    report_df = run_champion_report('data2023_2.csv')
    report_df = report_df.sort_values(by='언급횟수', ascending=False)
    print(report_df[['챔피언', '언급횟수', '감성', '분석결과']].head(20))
    report_df.to_csv('champion_final_analysis_report.csv', index=False, encoding='utf-8-sig')

NameError: name 'pos_keywords' is not defined

In [ ]:
import pandas as pd
import re
from collections import Counter

# 1. 172명 챔피언 리스트 및 감성 사전 정의 (생략 - 본문 동일)
# ... [이전 코드와 동일한 리스트 및 사전 설정] ...

def get_advanced_sentiment(comment_text):
    """댓글 하나의 감성 점수를 계산 (문맥 고려)"""
    words = re.sub(r'[^\w\s]', ' ', str(comment_text)).split()
    score = 0
    for word in words:
        if any(pk in word for pk in pos_keywords): score += 1
        if any(nk in word for nk in neg_keywords): score -= 1
    for i, word in enumerate(words):
        if any(cw in word for cw in neutral_context_words):
            context_window = " ".join(words[max(0, i-2) : min(len(words), i+3)])
            if any(pm in context_window for pm in pos_modifiers): score += 1
            if any(nm in context_window for nm in neg_modifiers): score -= 1
    return score

def is_target_mentioned(text, champ):
    """'진', '쉔' 등 1글자 챔피언의 오탐지를 방지하는 명사 판별 로직"""
    if len(champ) > 1:
        return champ in str(text).replace(' ', '')
    else:
        # 단어 앞뒤가 공백이거나 조사(이/가/은/는/을/를 등)가 붙은 명사 형태만 매칭
        pattern = f"(?:^|\\s){champ}(?=[이가은는을를의와과도만\\s]|$)"
        return bool(re.search(pattern, str(text)))

def run_champion_report(file_name):
    df = pd.read_csv(file_name)
    final_results = []
    for champ in champions_list:
        mask = df['comment_text'].apply(lambda x: is_target_mentioned(x, champ))
        champ_df = df[mask]
        if len(champ_df) < 5: continue
            
        total_score = sum(get_advanced_sentiment(t) for t in champ_df['comment_text'])
        all_words = []
        for text in champ_df['comment_text']:
            clean_words = re.sub(r'[^\w\s]', ' ', str(text)).split()
            all_words.extend([w for w in clean_words if champ not in w and len(w) > 1])
            
        keywords_str = ", ".join([f"{w}({c})" for w, c in Counter(all_words).most_common(3)])
        sentiment = "긍정적" if total_score > 0 else ("부정적" if total_score < 0 else "중립적")
        
        # 분석 결과 문장 생성
        thought = "유저들이 해당 챔피언의 디자인이나 성능 만족도, 혹은 재미 요소에 대해 좋은 평가를 내리고 있는 것으로 보입니다." if sentiment == "긍정적" else \
                  "유저들이 해당 챔피언의 밸런스 붕괴나 상대할 때의 불쾌감, 혹은 성능 하락에 대해 불만을 느끼고 있는 것으로 보입니다." if sentiment == "부정적" else \
                  "객관적인 패치 수치 공유나 일반적인 질문 사항으로 언급되고 있습니다."
            
        final_results.append({
            '챔피언': champ, '언급횟수': len(champ_df), '감성': sentiment,
            '분석결과': f"[{keywords_str}] 등의 이유로 챔피언을 언급했다. 그래서 내 생각은 {thought}"
        })
    return pd.DataFrame(final_results)

# --- 실행부 ---
if __name__ == "__main__":
    report_df = run_champion_report('data2023_1KR.csv')
    report_df = report_df.sort_values(by='언급횟수', ascending=False)
    print(report_df[['챔피언', '언급횟수', '감성', '분석결과']].head(20))
    report_df.to_csv('champion_final_analysis_report_kr.csv', index=False, encoding='utf-8-sig')

In [ ]:
import pandas as pd
import re
from collections import Counter

# 1. 분석 대상 포지션 8개 설정
positions_list = ['탑', '미드', '정글', '원딜', '서폿', '서포터', '정글러', '원딜러']

# 2. 감성 분석용 사전 정의
pos_keywords = ['갓', '버프', '스킨', '간지', '최고', '상향', '꿀잼', '재미', '예쁘다', '기대', '추천']
neg_keywords = ['너프', '삭제', '불쾌', '쓰레기', '하향', '노잼', '짜증', '극혐', '망겜', '어려움', '구림', '패배', '트롤']

# 문맥 분석용 중립 단어 및 수식어
neutral_context_words = ['성능', '밸런스', '딜', '탱', '계수']
pos_modifiers = ['좋은', '뛰어난', '압도적', '미친', '최고', '버프', '사기', 'OP', '만족', '유리', '좋아']
neg_modifiers = ['나쁜', '구린', '쓰레기', '하향', '너프', '엉망', '형편없는', '부족', '떨어지는', '불만', '낮은']

def get_advanced_sentiment(comment_text):
    """댓글의 문맥을 고려하여 감성 점수를 합산합니다."""
    words = re.sub(r'[^\w\s]', ' ', str(comment_text)).split()
    score = 0
    for word in words:
        if any(pk in word for pk in pos_keywords): score += 1
        if any(nk in word for nk in neg_keywords): score -= 1
    for i, word in enumerate(words):
        if any(cw in word for cw in neutral_context_words):
            # 단어 앞뒤 2단어 윈도우 분석
            context = " ".join(words[max(0, i-2) : min(len(words), i+3)])
            if any(pm in context for pm in pos_modifiers): score += 1
            if any(nm in context for nm in neg_modifiers): score -= 1
    return score

def is_position_mentioned(text, pos):
    """정규표현식을 이용해 독립된 명사로 쓰인 포지션만 필터링합니다."""
    # 단어 앞뒤가 공백이거나 조사가 붙은 경우만 인식하여 '정글'이 '정글러'에 포함되는 것을 방지
    pattern = f"(?:^|\\s){pos}(?=[이가은는을를의와과도만\\s]|$)"
    return bool(re.search(pattern, str(text)))

def run_position_report(file_name):
    df = pd.read_csv(file_name)
    results = []
    
    for pos in positions_list:
        mask = df['comment_text'].apply(lambda x: is_position_mentioned(x, pos))
        pos_df = df[mask]
        
        if len(pos_df) == 0: continue
            
        total_score = sum(get_advanced_sentiment(t) for t in pos_df['comment_text'])
        all_words = []
        for text in pos_df['comment_text']:
            clean_words = re.sub(r'[^\w\s]', ' ', str(text)).split()
            all_words.extend([w for w in clean_words if pos not in w and len(w) > 1])
        
        keywords = ", ".join([f"{w}({c})" for w, c in Counter(all_words).most_common(3)])
        sentiment = "긍정적" if total_score > 0 else ("부정적" if total_score < 0 else "중립적")
        
        # 내 생각(인사이트) 생성 로직
        if sentiment == "긍정적":
            thought = f"해당 포지션 유저들이 현재 메타나 아이템 변경에 만족하거나 재미를 느끼고 있습니다."
        elif sentiment == "부정적":
            thought = f"해당 포지션의 영향력이 낮거나 상대하기 까다로운 챔피언들 때문에 불만이 많은 상태입니다."
        else:
            thought = f"포지션 밸런스에 대해 큰 이슈 없이 일반적인 정보 공유가 이루어지고 있습니다."
            
        results.append({
            '포지션': pos,
            '언급횟수': len(pos_df),
            '감성': sentiment,
            '분석결과': f"[{keywords}] 등의 이유로 언급됨. 그래서 내 생각은 {thought}"
        })
        
    return pd.DataFrame(results)

# --- 메인 실행부 ---
if __name__ == "__main__":
    # 1. 파일 불러오기 및 분석 실행
    report_df = run_position_report('data2023_2.csv')
    
    # 2. 결과 출력 (상위 데이터 확인)
    print("\n=== 롤 포지션별 정밀 감성 분석 결과 ===")
    print(report_df[['포지션', '언급횟수', '감성']].to_string(index=False))
    
    # 3. CSV 파일 저장
    report_df.to_csv('position_final_analysis.csv', index=False, encoding='utf-8-sig')
    print("\n[완료] 상세 분석 보고서가 'position_final_analysis.csv'로 저장되었습니다.")

In [ ]:
import pandas as pd
import re
from collections import Counter

# 1. 분석 대상 포지션 8개 설정
positions_list = ['탑', '미드', '정글', '원딜', '서폿', '서포터', '정글러', '원딜러']

# 2. 감성 분석용 사전 정의
pos_keywords = ['갓', '버프', '스킨', '간지', '최고', '상향', '꿀잼', '재미', '예쁘다', '기대', '추천']
neg_keywords = ['너프', '삭제', '불쾌', '쓰레기', '하향', '노잼', '짜증', '극혐', '망겜', '어려움', '구림', '패배', '트롤']

# 문맥 분석용 중립 단어 및 수식어
neutral_context_words = ['성능', '밸런스', '딜', '탱', '계수']
pos_modifiers = ['좋은', '뛰어난', '압도적', '미친', '최고', '버프', '사기', 'OP', '만족', '유리', '좋아']
neg_modifiers = ['나쁜', '구린', '쓰레기', '하향', '너프', '엉망', '형편없는', '부족', '떨어지는', '불만', '낮은']

def get_advanced_sentiment(comment_text):
    """댓글의 문맥을 고려하여 감성 점수를 합산합니다."""
    words = re.sub(r'[^\w\s]', ' ', str(comment_text)).split()
    score = 0
    for word in words:
        if any(pk in word for pk in pos_keywords): score += 1
        if any(nk in word for nk in neg_keywords): score -= 1
    for i, word in enumerate(words):
        if any(cw in word for cw in neutral_context_words):
            # 단어 앞뒤 2단어 윈도우 분석
            context = " ".join(words[max(0, i-2) : min(len(words), i+3)])
            if any(pm in context for pm in pos_modifiers): score += 1
            if any(nm in context for nm in neg_modifiers): score -= 1
    return score

def is_position_mentioned(text, pos):
    """정규표현식을 이용해 독립된 명사로 쓰인 포지션만 필터링합니다."""
    # 단어 앞뒤가 공백이거나 조사가 붙은 경우만 인식하여 '정글'이 '정글러'에 포함되는 것을 방지
    pattern = f"(?:^|\\s){pos}(?=[이가은는을를의와과도만\\s]|$)"
    return bool(re.search(pattern, str(text)))

def run_position_report(file_name):
    df = pd.read_csv(file_name)
    results = []
    
    for pos in positions_list:
        mask = df['comment_text'].apply(lambda x: is_position_mentioned(x, pos))
        pos_df = df[mask]
        
        if len(pos_df) == 0: continue
            
        total_score = sum(get_advanced_sentiment(t) for t in pos_df['comment_text'])
        all_words = []
        for text in pos_df['comment_text']:
            clean_words = re.sub(r'[^\w\s]', ' ', str(text)).split()
            all_words.extend([w for w in clean_words if pos not in w and len(w) > 1])
        
        keywords = ", ".join([f"{w}({c})" for w, c in Counter(all_words).most_common(3)])
        sentiment = "긍정적" if total_score > 0 else ("부정적" if total_score < 0 else "중립적")
        
        # 내 생각(인사이트) 생성 로직
        if sentiment == "긍정적":
            thought = f"해당 포지션 유저들이 현재 메타나 아이템 변경에 만족하거나 재미를 느끼고 있습니다."
        elif sentiment == "부정적":
            thought = f"해당 포지션의 영향력이 낮거나 상대하기 까다로운 챔피언들 때문에 불만이 많은 상태입니다."
        else:
            thought = f"포지션 밸런스에 대해 큰 이슈 없이 일반적인 정보 공유가 이루어지고 있습니다."
            
        results.append({
            '포지션': pos,
            '언급횟수': len(pos_df),
            '감성': sentiment,
            '분석결과': f"[{keywords}] 등의 이유로 언급됨. 그래서 내 생각은 {thought}"
        })
        
    return pd.DataFrame(results)

# --- 메인 실행부 ---
if __name__ == "__main__":
    # 1. 파일 불러오기 및 분석 실행
    report_df = run_position_report('data2023_1KR.csv')
    
    # 2. 결과 출력 (상위 데이터 확인)
    print("\n=== 롤 포지션별 정밀 감성 분석 결과 ===")
    print(report_df[['포지션', '언급횟수', '감성']].to_string(index=False))
    
    # 3. CSV 파일 저장
    report_df.to_csv('position_final_analysis_kr.csv', index=False, encoding='utf-8-sig')
    print("\n[완료] 상세 분석 보고서가 'position_final_analysis_kr.csv'로 저장되었습니다.")

In [7]:
import pandas as pd
import re
from collections import Counter

# 1. 통합할 포지션 그룹 설정
position_groups = {
    '탑': ['탑'],
    '미드': ['미드'],
    '정글': ['정글', '정글러'],
    '원딜': ['원딜', '원딜러'],
    '서폿': ['서폿', '서포터']
}

# 2. 감성 사전 및 문맥 분석 함수 (생략 - 이전과 동일한 로직 사용)
# [get_advanced_sentiment 함수 및 사전 데이터 포함됨]

# 감성 분석용 사전 정의
pos_keywords = ['갓', '버프', '스킨', '간지', '최고', '상향', '꿀잼', '재미', '예쁘다', '기대', '추천']
neg_keywords = ['너프', '삭제', '불쾌', '쓰레기', '하향', '노잼', '짜증', '극혐', '망겜', '어려움', '구림', '패배', '트롤']

# 문맥 분석용 중립 단어 및 수식어
neutral_context_words = ['성능', '밸런스', '딜', '탱', '계수']
pos_modifiers = ['좋은', '뛰어난', '압도적', '미친', '최고', '버프', '사기', 'OP', '만족', '유리', '좋아']
neg_modifiers = ['나쁜', '구린', '쓰레기', '하향', '너프', '엉망', '형편없는', '부족', '떨어지는', '불만', '낮은']

def is_any_position_mentioned(text, pos_list):
    """그룹 내 키워드 중 하나라도 독립 명사로 쓰였는지 확인"""
    for pos in pos_list:
        pattern = f"(?:^|\\s){pos}(?=[이가은는을를의와과도만\\s]|$)"
        if bool(re.search(pattern, str(text))):
            return True
    return False

def run_grouped_position_report(file_name):
    df = pd.read_csv(file_name)
    results = []
    
    for group_name, keywords in position_groups.items():
        # 그룹 내 단어가 하나라도 포함된 댓글 필터링
        mask = df['comment_text'].apply(lambda x: is_any_position_mentioned(x, keywords))
        group_df = df[mask]
        
        if len(group_df) == 0: continue
            
        # 감성 점수 통합 계산
        total_score = sum(get_advanced_sentiment(t) for t in group_df['comment_text'])
        
        # 그룹 통합 키워드 추출
        all_words = []
        for text in group_df['comment_text']:
            clean_words = re.sub(r'[^\w\s]', ' ', str(text)).split()
            all_words.extend([w for w in clean_words if not any(k in w for k in keywords) and len(w) > 1])
        
        keywords_str = ", ".join([f"{w}({c})" for w, c in Counter(all_words).most_common(3)])
        sentiment = "긍정적" if total_score > 0 else ("부정적" if total_score < 0 else "중립적")
        
        # 통합 인사이트 생성
        thought = f"해당 포지션({'+'.join(keywords)})의 전반적인 메타 불만이 감지됩니다." if sentiment == "부정적" else \
                  f"해당 포지션 유저들이 현재 변화에 만족하고 있습니다."
            
        results.append({
            '포지션 그룹': f"{group_name} ({'+'.join(keywords)})",
            '언급횟수': len(group_df),
            '감성': sentiment,
            '분석결과': f"[{keywords_str}] 등의 이유로 언급됨. 그래서 내 생각은 {thought}"
        })
        
    return pd.DataFrame(results)

# --- 실행부 ---
if __name__ == "__main__":
    report = run_grouped_position_report('data2023_1KR.csv')
    print(report[['포지션 그룹', '언급횟수', '감성']].to_string(index=False))
    report.to_csv('position_grouped_analysis_kr.csv', index=False, encoding='utf-8-sig')

     포지션 그룹  언급횟수  감성
      탑 (탑)   499 부정적
    미드 (미드)   368 부정적
정글 (정글+정글러)   725 부정적
원딜 (원딜+원딜러)   401 긍정적
서폿 (서폿+서포터)   403 부정적


In [9]:
import pandas as pd
import re
from collections import Counter

# 1. 통합할 포지션 그룹 설정
position_groups = {
    '탑': ['탑'],
    '미드': ['미드'],
    '정글': ['정글', '정글러'],
    '원딜': ['원딜', '원딜러'],
    '서폿': ['서폿', '서포터']
}

# 2. [사용자 요청] 감성 분석용 사전 정의 및 수식어
pos_keywords = ['갓', '버프', '스킨', '간지', '최고', '상향', '꿀잼', '재미', '예쁘다', '기대', '추천']
neg_keywords = ['너프', '삭제', '불쾌', '쓰레기', '하향', '노잼', '짜증', '극혐', '망겜', '어려움', '구림', '패배', '트롤']

neutral_context_words = ['성능', '밸런스', '딜', '탱', '계수']
pos_modifiers = ['좋은', '뛰어난', '압도적', '미친', '최고', '버프', '사기', 'OP', '만족', '유리', '좋아']
neg_modifiers = ['나쁜', '구린', '쓰레기', '하향', '너프', '엉망', '형편없는', '부족', '떨어지는', '불만', '낮은']

def get_advanced_sentiment(comment_text):
    """문맥(Window)을 고려한 고도화 감성 점수 계산"""
    words = re.sub(r'[^\w\s]', ' ', str(comment_text)).split()
    score = 0
    # 단어별 점수 (1단계)
    for word in words:
        if any(pk in word for pk in pos_keywords): score += 1
        if any(nk in word for nk in neg_keywords): score -= 1
    # 주변 수식어 분석 (2단계)
    for i, word in enumerate(words):
        if any(cw in word for cw in neutral_context_words):
            context = " ".join(words[max(0, i-2) : min(len(words), i+3)])
            if any(pm in context for pm in pos_modifiers): score += 1
            if any(nm in context for nm in neg_modifiers): score -= 1
    return score

# ... [is_any_position_mentioned 함수 및 run_grouped_position_report 로직 동일] ...

if __name__ == "__main__":
    report_df = run_grouped_position_report('data2023_1KR.csv')
    report_df.to_csv('position_grouped_final_report_kr.csv', index=False, encoding='utf-8-sig')
    print(report_df[['포지션 그룹', '언급횟수', '감성']].head())

        포지션 그룹  언급횟수   감성
0        탑 (탑)   499  부정적
1      미드 (미드)   368  부정적
2  정글 (정글+정글러)   725  부정적
3  원딜 (원딜+원딜러)   401  긍정적
4  서폿 (서폿+서포터)   403  부정적


In [12]:
import pandas as pd
import re
from collections import Counter

# 1. 통합할 포지션 그룹 설정
position_groups = {
    '탑': ['탑'],
    '미드': ['미드'],
    '정글': ['정글', '정글러'],
    '원딜': ['원딜', '원딜러'],
    '서폿': ['서폿', '서포터']
}

# 2. 감성 및 문맥 분석용 사전
pos_keywords = ['갓', '버프', '스킨', '간지', '최고', '상향', '꿀잼', '재미', '예쁘다', '기대', '추천']
neg_keywords = ['너프', '삭제', '불쾌', '쓰레기', '하향', '노잼', '짜증', '극혐', '망겜', '어려움', '구림', '패배', '트롤']
neutral_context_words = ['성능', '밸런스', '딜', '탱', '계수']
pos_modifiers = ['좋은', '뛰어난', '압도적', '미친', '최고', '버프', '사기', 'OP', '만족', '유리', '좋아']
neg_modifiers = ['나쁜', '구린', '쓰레기', '하향', '너프', '엉망', '형편없는', '부족', '떨어지는', '불만', '낮은']

# [핵심] 불만 원인 분석용 카테고리 버킷
reason_buckets = {
    '밸런스 조정': ['너프', '버프', '상향', '하향', '삭제', '수치', '조정', '계수'],
    '플레이 경험': ['불쾌', '짜증', '극혐', '노잼', '꿀잼', '재미', '트롤', '쓰레기'],
    '아이템 및 성능': ['아이템', '템', '성능', '사기', 'OP', '스탯', '빌드'],
    '영향력 및 메타': ['영향력', '주도권', '라인', '메타', '시즌', '로밍', '한타', '오브젝트']
}

# 3. 분석용 함수 정의 (Sentiment, Mention, Factor 분석)
def get_advanced_sentiment(text):
    words = re.sub(r'[^\w\s]', ' ', str(text)).split()
    score = 0
    for word in words:
        if any(pk in word for pk in pos_keywords): score += 1
        if any(nk in word for nk in neg_keywords): score -= 1
    for i, word in enumerate(words):
        if any(cw in word for cw in neutral_context_words):
            window = " ".join(words[max(0, i-2) : min(len(words), i+3)])
            if any(pm in window for pm in pos_modifiers): score += 1
            if any(nm in window for nm in neg_modifiers): score -= 1
    return score

def is_target_mentioned(text, target):
    pattern = f"(?:^|\\s){target}(?=[이가은는을를의와과도만\\s]|$)"
    return bool(re.search(pattern, str(text)))

def analyze_factors(comments, keywords_to_exclude):
    all_text = " ".join(comments)
    words = re.sub(r'[^\w\s]', ' ', all_text).split()
    filtered = [w for w in words if not any(k in w for k in keywords_to_exclude) and len(w) > 1]
    cat_counts = Counter()
    for w in filtered:
        for cat, bucket in reason_buckets.items():
            if any(b in w for b in bucket): cat_counts[cat] += 1
    return ", ".join([w for w, c in Counter(filtered).most_common(3)]), [cat for cat, c in cat_counts.most_common(2)]

# 4. 실행 및 리포트 생성
def run_pos_report(file_name):
    df = pd.read_csv(file_name)
    results = []
    for group_name, keywords in position_groups.items():
        mask = df['comment_text'].apply(lambda x: any(is_target_mentioned(x, k) for k in keywords))
        pos_df = df[mask]
        if len(pos_df) == 0: continue
        
        comments = pos_df['comment_text'].tolist()
        score = sum(get_advanced_sentiment(t) for t in comments)
        top_words, top_cats = analyze_factors(comments, keywords)
        sentiment = "긍정적" if score > 0 else ("부정적" if score < 0 else "중립적")
        
        if sentiment == "부정적":
            reason_msg = f" 특히 '{', '.join(top_cats)}' 관련 키워드가 지배적입니다." if top_cats else ""
            thought = f"해당 포지션 유저들은 [{top_words}] 등에서 큰 피로감을 느끼고 있습니다.{reason_msg} 영향력 배분에 대한 재검토가 필요해 보입니다."
        else:
            thought = f"[{top_words}] 위주로 언급되며, 비교적 안정적인 여론을 형성하고 있습니다."
            
        results.append({'포지션': group_name, '언급횟수': len(pos_df), '감성': sentiment, 
                        '분석결과': f"[{top_words}] 등의 이유로 언급됨. 그래서 내 생각은 {thought}"})
    return pd.DataFrame(results)

# 결과 출력
pos_report = run_pos_report('data2023_2.csv')
print(pos_report[['포지션', '언급횟수', '감성', '분석결과']])
pos_report.to_csv('final_position_report_eng.csv', index=False, encoding='utf-8-sig')

  포지션  언급횟수   감성                                               분석결과
0   탑   180  부정적  [라인, 너무, 정말] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들은...
1  미드    98  부정적  [너무, 있는, 정말] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들은...
2  정글   357  부정적  [너무, 정말, 저는] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들은...
3  원딜   134  부정적  [정말, 이제, 아이템] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들...
4  서폿   137  부정적  [정말, 때문에, 있는] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들...


In [16]:
import pandas as pd
import re
from collections import Counter

# 1. 통합할 포지션 그룹 설정
position_groups = {
    '탑': ['탑'],
    '미드': ['미드'],
    '정글': ['정글', '정글러'],
    '원딜': ['원딜', '원딜러'],
    '서폿': ['서폿', '서포터']
}

# 2. 감성 및 문맥 분석용 사전
pos_keywords = ['갓', '버프', '스킨', '간지', '최고', '상향', '꿀잼', '재미', '예쁘다', '기대', '추천']
neg_keywords = ['너프', '삭제', '불쾌', '쓰레기', '하향', '노잼', '짜증', '극혐', '망겜', '어려움', '구림', '패배', '트롤']
neutral_context_words = ['성능', '밸런스', '딜', '탱', '계수']
pos_modifiers = ['좋은', '뛰어난', '압도적', '미친', '최고', '버프', '사기', 'OP', '만족', '유리', '좋아']
neg_modifiers = ['나쁜', '구린', '쓰레기', '하향', '너프', '엉망', '형편없는', '부족', '떨어지는', '불만', '낮은']

# [핵심] 불만 원인 분석용 카테고리 버킷
reason_buckets = {
    '밸런스 조정': ['너프', '버프', '상향', '하향', '삭제', '수치', '조정', '계수'],
    '플레이 경험': ['불쾌', '짜증', '극혐', '노잼', '꿀잼', '재미', '트롤', '쓰레기'],
    '아이템 및 성능': ['아이템', '템', '성능', '사기', 'OP', '스탯', '빌드'],
    '영향력 및 메타': ['영향력', '주도권', '라인', '메타', '시즌', '로밍', '한타', '오브젝트']
}

# [추가] 분석에서 제외할 불용어(Stopwords) 리스트
stopwords = ['진짜', '너무', '그냥', '아니', '이거', '정말', '존나', '많이', '계속', '좀', '수', '있는', '하는', '그리고', '근데', '어떻게', 'ㅋㅋ']

# 3. 분석용 함수 정의 (Sentiment, Mention, Factor 분석)
def get_advanced_sentiment(text):
    words = re.sub(r'[^\w\s]', ' ', str(text)).split()
    score = 0
    for word in words:
        if any(pk in word for pk in pos_keywords): score += 1
        if any(nk in word for nk in neg_keywords): score -= 1
        
    for i, word in enumerate(words):
        if any(cw in word for cw in neutral_context_words):
            window = " ".join(words[max(0, i-2) : min(len(words), i+3)])
            if any(pm in window for pm in pos_modifiers): score += 1
            if any(nm in window for nm in neg_modifiers): score -= 1
    return score

def is_target_mentioned(text, target):
    # 한국어 조사 처리 (이가은는을를의와과도만)
    pattern = f"(?:^|\\s){target}(?=[이가은는을를의와과도만\\s]|$)"
    return bool(re.search(pattern, str(text)))

def analyze_factors(comments, keywords_to_exclude):
    all_text = " ".join(comments)
    words = re.sub(r'[^\w\s]', ' ', all_text).split()
    
    # 불용어(stopwords) 및 타겟 키워드, 한 글자 단어 제외
    filtered = [
        w for w in words 
        if not any(k in w for k in keywords_to_exclude) 
        and not any(s == w for s in stopwords)
        and len(w) > 1
    ]
    
    cat_counts = Counter()
    for w in filtered:
        for cat, bucket in reason_buckets.items():
            if any(b in w for b in bucket): 
                cat_counts[cat] += 1
                
    # 자주 등장한 단어 Top 3 및 상위 2개 카테고리 도출
    top_words = [w for w, c in Counter(filtered).most_common(3)]
    top_cats = [cat for cat, c in cat_counts.most_common(2) if c > 0]
    
    return ", ".join(top_words), top_cats

# 4. 실행 및 리포트 생성
def run_pos_report(file_name):
    try:
        df = pd.read_csv(file_name)
    except FileNotFoundError:
        print(f"오류: '{file_name}' 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
        return pd.DataFrame()

    results = []
    for group_name, keywords in position_groups.items():
        mask = df['comment_text'].apply(lambda x: any(is_target_mentioned(x, k) for k in keywords))
        pos_df = df[mask]
        
        if len(pos_df) == 0: 
            continue
        
        comments = pos_df['comment_text'].tolist()
        score = sum(get_advanced_sentiment(t) for t in comments)
        top_words_str, top_cats = analyze_factors(comments, keywords)
        
        sentiment = "긍정적" if score > 0 else ("부정적" if score < 0 else "중립적")
        
        # 리포트 코멘트(Thought) 생성
        if sentiment == "부정적":
            reason_msg = f" 특히 '{', '.join(top_cats)}' 관련 키워드가 지배적입니다." if top_cats else ""
            thought = f"해당 포지션 유저들은 [{top_words_str}] 등에서 큰 피로감을 느끼고 있습니다.{reason_msg} 영향력 배분에 대한 재검토가 필요해 보입니다."
        else:
            thought = f"[{top_words_str}] 위주로 언급되며, 비교적 안정적인 여론을 형성하고 있습니다."
            
        results.append({
            '포지션': group_name, 
            '언급횟수': len(pos_df), 
            '감성': sentiment, 
            '분석결과': f"[{top_words_str}] 등의 이유로 언급됨. 그래서 내 생각은 {thought}"
        })
        
    return pd.DataFrame(results)

# 5. 결과 출력 및 저장
if __name__ == "__main__":
    file_path = 'data2023_1KR.csv'
    pos_report = run_pos_report(file_path)
    
    if not pos_report.empty:
        print(pos_report[['포지션', '언급횟수', '감성', '분석결과']])
        pos_report.to_csv('final_position_report_kr.csv', index=False, encoding='utf-8-sig')
        print(f"\n✅ 분석이 완료되어 'final_position_report_kr.csv' 파일로 저장되었습니다.")

  포지션  언급횟수   감성                                               분석결과
0   탑   499  부정적  [미드, 정글, 원딜] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들은...
1  미드   368  부정적  [정글, 서폿, 바텀] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들은...
2  정글   725  부정적  [미드, 너프, 라인] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들은...
3  원딜   401  긍정적  [미드, 서폿, 정글] 등의 이유로 언급됨. 그래서 내 생각은 [미드, 서폿, 정글...
4  서폿   403  부정적  [정글, 원딜, 미드] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들은...

✅ 분석이 완료되어 'final_position_report_kr.csv' 파일로 저장되었습니다.


In [9]:
import pandas as pd
import re
from collections import Counter

# 1. 통합할 포지션 그룹 설정
position_groups = {
    '탑': ['탑'],
    '미드': ['미드'],
    '정글': ['정글', '정글러'],
    '원딜': ['원딜', '원딜러'],
    '서폿': ['서폿', '서포터']
}

# 2. 감성 및 문맥 분석용 사전
pos_keywords = ['갓', '버프', '스킨', '간지', '최고', '상향', '꿀잼', '재미', '예쁘다', '기대', '추천']
neg_keywords = ['너프', '삭제', '불쾌', '쓰레기', '하향', '노잼', '짜증', '극혐', '망겜', '어려움', '구림', '패배', '트롤']
neutral_context_words = ['성능', '밸런스', '딜', '탱', '계수']
pos_modifiers = ['좋은', '뛰어난', '압도적', '미친', '최고', '버프', '사기', 'OP', '만족', '유리', '좋아']
neg_modifiers = ['나쁜', '구린', '쓰레기', '하향', '너프', '엉망', '형편없는', '부족', '떨어지는', '불만', '낮은']

# [핵심] 불만 원인 분석용 카테고리 버킷
reason_buckets = {
    '밸런스 조정': ['너프', '버프', '상향', '하향', '삭제', '수치', '조정', '계수'],
    '플레이 경험': ['불쾌', '짜증', '극혐', '노잼', '꿀잼', '재미', '트롤', '쓰레기'],
    '아이템 및 성능': ['아이템', '템', '성능', '사기', 'OP', '스탯', '빌드'],
    '영향력 및 메타': ['영향력', '주도권', '라인', '메타', '시즌', '로밍', '한타', '오브젝트', '포지션']
}

# [추가] 분석에 포함할 필수 키워드 리스트 (화이트리스트)
# 분석 목적에 맞게 필요한 단어들을 자유롭게 추가/수정하시면 됩니다.
target_keywords = [
    '라인', '포지션', '아이템', '메타', '오브젝트', '한타', '스탯', '빌드', 
    '주도권', '로밍', '너프', '버프', '상향', '하향', '사기', '쓰레기', '트롤', 
    '밸런스', '성능', '딜', '탱', '초반', '후반', '캐리','탑','정글','미드','원딜','서폿'
]

# 3. 분석용 함수 정의 (Sentiment, Mention, Factor 분석)
def get_advanced_sentiment(text):
    words = re.sub(r'[^\w\s]', ' ', str(text)).split()
    score = 0
    for word in words:
        if any(pk in word for pk in pos_keywords): score += 1
        if any(nk in word for nk in neg_keywords): score -= 1
        
    for i, word in enumerate(words):
        if any(cw in word for cw in neutral_context_words):
            window = " ".join(words[max(0, i-2) : min(len(words), i+3)])
            if any(pm in window for pm in pos_modifiers): score += 1
            if any(nm in window for nm in neg_modifiers): score -= 1
    return score

def is_target_mentioned(text, target):
    # 한국어 조사 처리 (이가은는을를의와과도만)
    pattern = f"(?:^|\\s){target}(?=[이가은는을를의와과도만\\s]|$)"
    return bool(re.search(pattern, str(text)))

def analyze_factors(comments, keywords_to_exclude):
    all_text = " ".join(comments)
    words = re.sub(r'[^\w\s]', ' ', all_text).split()
    
    # [수정] 불용어 제외 로직을 화이트리스트 포함 로직으로 변경
    filtered = []
    for w in words:
        # 1. 포지션명 자체(예: 미드, 탑)는 결과에서 제외
        if any(k in w for k in keywords_to_exclude):
            continue
        # 2. 내가 지정한 필수 키워드(target_keywords)가 포함된 단어만 수집
        if any(t in w for t in target_keywords):
            filtered.append(w)
    
    cat_counts = Counter()
    for w in filtered:
        for cat, bucket in reason_buckets.items():
            if any(b in w for b in bucket): 
                cat_counts[cat] += 1
                
    # 자주 등장한 단어 Top 3 및 상위 2개 카테고리 도출
    top_words = [w for w, c in Counter(filtered).most_common(3)]
    top_cats = [cat for cat, c in cat_counts.most_common(2) if c > 0]
    
    return ", ".join(top_words), top_cats

# 4. 실행 및 리포트 생성
def run_pos_report(file_name):
    try:
        df = pd.read_csv(file_name)
    except FileNotFoundError:
        print(f"오류: '{file_name}' 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
        return pd.DataFrame()

    results = []
    for group_name, keywords in position_groups.items():
        mask = df['comment_text'].apply(lambda x: any(is_target_mentioned(x, k) for k in keywords))
        pos_df = df[mask]
        
        if len(pos_df) == 0: 
            continue
        
        comments = pos_df['comment_text'].tolist()
        score = sum(get_advanced_sentiment(t) for t in comments)
        top_words_str, top_cats = analyze_factors(comments, keywords)
        
        sentiment = "긍정적" if score > 0 else ("부정적" if score < 0 else "중립적")
        
        # 리포트 코멘트(Thought) 생성
        if sentiment == "부정적":
            reason_msg = f" 특히 '{', '.join(top_cats)}' 관련 키워드가 지배적입니다." if top_cats else ""
            thought = f"해당 포지션 유저들은 [{top_words_str}] 등에서 큰 피로감을 느끼고 있습니다.{reason_msg} 영향력 배분에 대한 재검토가 필요해 보입니다."
        else:
            thought = f"[{top_words_str}] 위주로 언급되며, 비교적 안정적인 여론을 형성하고 있습니다."
            
        results.append({
            '포지션': group_name, 
            '언급횟수': len(pos_df), 
            '감성': sentiment, 
            '분석결과': f"[{top_words_str}] 등의 이유로 언급됨. 그래서 내 생각은 {thought}"
        })
        
    return pd.DataFrame(results)

# 5. 결과 출력 및 저장
if __name__ == "__main__":
    file_path = 'data2023_1KR.csv'
    pos_report = run_pos_report(file_path)
    
    if not pos_report.empty:
        print(pos_report[['포지션', '언급횟수', '감성', '분석결과']])
        pos_report.to_csv('final_position_report_kr.csv', index=False, encoding='utf-8-sig')
        print(f"\n✅ 분석이 완료되어 'final_position_report_kr.csv' 파일로 저장되었습니다.")

  포지션  언급횟수   감성                                               분석결과
0   탑   499  부정적  [미드, 정글, 원딜] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들은...
1  미드   368  부정적  [탑, 정글, 서폿] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들은 ...
2  정글   725  부정적  [탑, 미드, 너프] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들은 ...
3  원딜   401  긍정적  [탑, 미드, 서폿] 등의 이유로 언급됨. 그래서 내 생각은 [탑, 미드, 서폿] ...
4  서폿   403  부정적  [정글, 탑, 원딜] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들은 ...

✅ 분석이 완료되어 'final_position_report_kr.csv' 파일로 저장되었습니다.


In [8]:
import pandas as pd
import re
from collections import Counter

# 1. 통합할 포지션 그룹 설정
position_groups = {
    '탑': ['탑'],
    '미드': ['미드'],
    '정글': ['정글', '정글러'],
    '원딜': ['원딜', '원딜러'],
    '서폿': ['서폿', '서포터']
}

# 2. 감성 및 문맥 분석용 사전
pos_keywords = ['갓', '버프', '스킨', '간지', '최고', '상향', '꿀잼', '재미', '예쁘다', '기대', '추천']
neg_keywords = ['너프', '삭제', '불쾌', '쓰레기', '하향', '노잼', '짜증', '극혐', '망겜', '어려움', '구림', '패배', '트롤']
neutral_context_words = ['성능', '밸런스', '딜', '탱', '계수']
pos_modifiers = ['좋은', '뛰어난', '압도적', '미친', '최고', '버프', '사기', 'OP', '만족', '유리', '좋아']
neg_modifiers = ['나쁜', '구린', '쓰레기', '하향', '너프', '엉망', '형편없는', '부족', '떨어지는', '불만', '낮은']

# [핵심] 불만 원인 분석용 카테고리 버킷
reason_buckets = {
    '밸런스 조정': ['너프', '버프', '상향', '하향', '삭제', '수치', '조정', '계수'],
    '플레이 경험': ['불쾌', '짜증', '극혐', '노잼', '꿀잼', '재미', '트롤', '쓰레기'],
    '아이템 및 성능': ['아이템', '템', '성능', '사기', 'OP', '스탯', '빌드'],
    '영향력 및 메타': ['영향력', '주도권', '라인', '메타', '시즌', '로밍', '한타', '오브젝트', '포지션']
}

# [추가] 분석에 포함할 필수 키워드 리스트 (화이트리스트)
# 분석 목적에 맞게 필요한 단어들을 자유롭게 추가/수정하시면 됩니다.
target_keywords = [
    '라인', '포지션', '아이템', '메타', '오브젝트', '한타', '스탯', '빌드', 
    '주도권', '로밍', '너프', '버프', '상향', '하향', '사기', '쓰레기', '트롤', 
    '밸런스', '성능', '딜', '탱', '초반', '후반', '캐리','탑','정글','미드','원딜','서폿'
]

# 3. 분석용 함수 정의 (Sentiment, Mention, Factor 분석)
def get_advanced_sentiment(text):
    words = re.sub(r'[^\w\s]', ' ', str(text)).split()
    score = 0
    for word in words:
        if any(pk in word for pk in pos_keywords): score += 1
        if any(nk in word for nk in neg_keywords): score -= 1
        
    for i, word in enumerate(words):
        if any(cw in word for cw in neutral_context_words):
            window = " ".join(words[max(0, i-2) : min(len(words), i+3)])
            if any(pm in window for pm in pos_modifiers): score += 1
            if any(nm in window for nm in neg_modifiers): score -= 1
    return score

def is_target_mentioned(text, target):
    # 한국어 조사 처리 (이가은는을를의와과도만)
    pattern = f"(?:^|\\s){target}(?=[이가은는을를의와과도만\\s]|$)"
    return bool(re.search(pattern, str(text)))

def analyze_factors(comments, keywords_to_exclude):
    all_text = " ".join(comments)
    words = re.sub(r'[^\w\s]', ' ', all_text).split()
    
    # [수정] 불용어 제외 로직을 화이트리스트 포함 로직으로 변경
    filtered = []
    for w in words:
        # 1. 포지션명 자체(예: 미드, 탑)는 결과에서 제외
        if any(k in w for k in keywords_to_exclude):
            continue
        # 2. 내가 지정한 필수 키워드(target_keywords)가 포함된 단어만 수집
        if any(t in w for t in target_keywords):
            filtered.append(w)
    
    cat_counts = Counter()
    for w in filtered:
        for cat, bucket in reason_buckets.items():
            if any(b in w for b in bucket): 
                cat_counts[cat] += 1
                
    # 자주 등장한 단어 Top 3 및 상위 2개 카테고리 도출
    top_words = [w for w, c in Counter(filtered).most_common(3)]
    top_cats = [cat for cat, c in cat_counts.most_common(2) if c > 0]
    
    return ", ".join(top_words), top_cats

# 4. 실행 및 리포트 생성
def run_pos_report(file_name):
    try:
        df = pd.read_csv(file_name)
    except FileNotFoundError:
        print(f"오류: '{file_name}' 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
        return pd.DataFrame()

    results = []
    for group_name, keywords in position_groups.items():
        mask = df['comment_text'].apply(lambda x: any(is_target_mentioned(x, k) for k in keywords))
        pos_df = df[mask]
        
        if len(pos_df) == 0: 
            continue
        
        comments = pos_df['comment_text'].tolist()
        score = sum(get_advanced_sentiment(t) for t in comments)
        top_words_str, top_cats = analyze_factors(comments, keywords)
        
        sentiment = "긍정적" if score > 0 else ("부정적" if score < 0 else "중립적")
        
        # 리포트 코멘트(Thought) 생성
        if sentiment == "부정적":
            reason_msg = f" 특히 '{', '.join(top_cats)}' 관련 키워드가 지배적입니다." if top_cats else ""
            thought = f"해당 포지션 유저들은 [{top_words_str}] 등에서 큰 피로감을 느끼고 있습니다.{reason_msg} 영향력 배분에 대한 재검토가 필요해 보입니다."
        else:
            thought = f"[{top_words_str}] 위주로 언급되며, 비교적 안정적인 여론을 형성하고 있습니다."
            
        results.append({
            '포지션': group_name, 
            '언급횟수': len(pos_df), 
            '감성': sentiment, 
            '분석결과': f"[{top_words_str}] 등의 이유로 언급됨. 그래서 내 생각은 {thought}"
        })
        
    return pd.DataFrame(results)

# 5. 결과 출력 및 저장
if __name__ == "__main__":
    file_path = 'data2023_2.csv'
    pos_report = run_pos_report(file_path)
    
    if not pos_report.empty:
        print(pos_report[['포지션', '언급횟수', '감성', '분석결과']])
        pos_report.to_csv('final_position_report_eng.csv', index=False, encoding='utf-8-sig')
        print(f"\n✅ 분석이 완료되어 'final_position_report_kr.csv' 파일로 저장되었습니다.")

  포지션  언급횟수   감성                                               분석결과
0   탑   180  부정적  [라인, 라인에서, 정글] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저...
1  미드    98  부정적  [라인, 라인에서, 탑] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들...
2  정글   357  부정적  [탑, 라인, 너프는] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들은...
3  원딜   134  부정적  [아이템, 탑, 탱커] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저들은...
4  서폿   137  부정적  [아이템을, 라인, 정글] 등의 이유로 언급됨. 그래서 내 생각은 해당 포지션 유저...

✅ 분석이 완료되어 'final_position_report_kr.csv' 파일로 저장되었습니다.
